# **Imports**

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src' / 'M2F').exists() and (p / 'untracked').exists():
            return p
    raise RuntimeError('Could not find repo root containing src/M2F and untracked')

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import shutil
from M2F.pyg_data_interfaces import DatasetInput, ProteinGraphInMemoryDataset
from M2F.embedding_utils import AAChainEmbedder
from M2F.cleaning_utils import clean_col
from M2F.feature_engineering_utils import encode_go, embed_AAsequences

/Users/yehormishchyriak/Desktop/BonhamLab/microbiome2function/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/yehormishchyriak/Desktop/BonhamLab/microbiome2function/src/M2F/dependencies/go-basic.obo: fmt(1.2) rel(2025-07-22) 43,230 Terms


# **CONFIG**

In [2]:
GO_DEPTH = 5
AA_MODEL_KEY = 'esm2_t6_8M_UR50D'
AA_BATCH_SIZE = 16
FORCE_RELOAD = False

subset_dir = REPO_ROOT / 'untracked' / 'test_data_subset'
out_root = REPO_ROOT / 'untracked' / 'prot1_real_uniprot_go_depth'
if FORCE_RELOAD:
    shutil.rmtree(out_root, ignore_errors=True)

# **Transforms**

In [3]:
go_label_map: dict[str, int] = {}
aa_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
aa_encoder = AAChainEmbedder(model_key=AA_MODEL_KEY, device=aa_device)


def _as_go_multihot(idx_tuple, y_dim: int):
    if not isinstance(idx_tuple, tuple) or y_dim == 0:
        return np.nan
    vec = np.zeros(y_dim, dtype=np.float32)
    if idx_tuple:
        vec[np.asarray(idx_tuple, dtype=np.int64)] = 1.0
    return vec if vec.sum() > 0 else np.nan


def composed_pre_transform(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()

    # M2F extraction/normalization.
    df = clean_col(df, 'Sequence', apply_norm=False, apply_strip_pubmed=False, inplace=True)
    df = clean_col(
        df,
        'Gene Ontology (molecular function)',
        apply_norm=False,
        apply_strip_pubmed=True,
        inplace=True,
    )

    # M2F GO depth encoding.
    df, labels = encode_go(
        df,
        col_name='Gene Ontology (molecular function)',
        depth=GO_DEPTH,
        inplace=True,
    )
    go_label_map.clear()
    go_label_map.update(labels)
    y_dim = len(go_label_map)

    # Fixed-width target vectors for tensor stacking.
    df.loc[:, 'Gene Ontology (molecular function)'] = df[
        'Gene Ontology (molecular function)'
    ].map(lambda t: _as_go_multihot(t, y_dim))

    # M2F AA sequence embedding.
    df = embed_AAsequences(df, embedder=aa_encoder, batch_size=AA_BATCH_SIZE, inplace=True)

    return df


def pre_filter_mask(df: pd.DataFrame):
    x_ok = df['Sequence'].map(
        lambda x: isinstance(x, np.ndarray) and x.size > 0 and np.isfinite(x).all()
    )
    y_ok = df['Gene Ontology (molecular function)'].map(
        lambda y: isinstance(y, np.ndarray) and y.size > 0 and np.isfinite(y).all() and (y.sum() > 0)
    )
    return x_ok & y_ok

# **Data**

In [4]:
dataset_input = DatasetInput(
    path_to_accession_ids_csv_file=subset_dir / 'uniref_index_count.csv',
    path_to_edge_csv_dir=subset_dir,
    X={'sequence': 'Sequence'},
    Y={'go_f': 'Gene Ontology (molecular function)'},
    edge_dst_column='j',
    edge_attr_columns=('v',),
    request_size=25,
    rps=1,
    max_retry=20,
)

ds = ProteinGraphInMemoryDataset(
    root=out_root,
    dataset_input=dataset_input,
    pre_transform=composed_pre_transform,
    pre_filter=pre_filter_mask,
    force_reload=FORCE_RELOAD,
)

data = ds[0]
print(f'nodes={data.num_nodes}, edges={data.num_edges}')
print(f'x={tuple(data.x.shape)}, y={tuple(data.y.shape)}, edge_attr={tuple(data.edge_attr.shape)}')

assert data.num_nodes > 0
assert data.edge_index.shape[0] == 2
assert data.x.shape[0] == data.num_nodes
assert data.y.shape[0] == data.num_nodes
if go_label_map:
    assert data.y.shape[1] == len(go_label_map)


nodes=65, edges=162
x=(65, 320), y=(65, 22), edge_attr=(162, 1)


# **Split Masks**

In [5]:
n_nodes = data.num_nodes
g = torch.Generator().manual_seed(1)
perm = torch.randperm(n_nodes, generator=g)

n_train = int(0.70 * n_nodes)
n_val = int(0.15 * n_nodes)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_mask = torch.zeros(n_nodes, dtype=torch.bool)
val_mask = torch.zeros(n_nodes, dtype=torch.bool)
test_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f'train={train_mask.sum().item()} val={val_mask.sum().item()} test={test_mask.sum().item()}')


train=45 val=9 test=11


# **GNN**

In [7]:
from M2F.gnn import GraphConvNodeClassifier

input_feature_dim = data.x.shape[1]
input_feature_dim = data.x.shape[1]
msg_dim = 128
state_dim = 128
out_classes = data.y.shape[1]

model = GraphConvNodeClassifier(
    in_dim=input_feature_dim,
    edge_dim=1,
    msg_dim=msg_dim,
    state_dim=state_dim,
    out_dim=out_classes
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()

# **Eval**

In [8]:
def accuracy(logits, y_true, mask, threshold=0.5):
    if mask.sum() == 0:
        return 0.0
    probs = torch.sigmoid(logits[mask])
    preds = (probs >= threshold).float()
    true = y_true[mask]
    return (preds == true).float().mean().item()

def recall(logits, y_true, mask, threshold=0.5, eps=1e-8):
    if mask.sum() == 0:
        return 0.0
    probs = torch.sigmoid(logits[mask])
    preds = (probs >= threshold).float()
    true = y_true[mask]
    tp = ((preds == 1) & (true == 1)).sum().float()
    fn = ((preds == 0) & (true == 1)).sum().float()
    return (tp / (tp + fn + eps)).item()


# **Loop**

In [11]:
EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()

    if epoch == 1 or epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_attr)
            val_acc = accuracy(logits, data.y, val_mask)
            test_acc = accuracy(logits, data.y, test_mask)
            val_recall = recall(logits, data.y, val_mask)
            test_recall = recall(logits, data.y, test_mask)
        print(f"Epoch {epoch:03d} | loss {loss.item():.4f} | val acc {val_acc:.3f} | test acc {test_acc:.3f} | val recall {val_recall:.3f} | test recall {test_recall:.3f}")


Epoch 001 | loss 0.1917 | val acc 0.939 | test acc 0.897 | val recall 0.471 | test recall 0.208
Epoch 005 | loss 0.1675 | val acc 0.934 | test acc 0.897 | val recall 0.471 | test recall 0.250
Epoch 010 | loss 0.1449 | val acc 0.934 | test acc 0.901 | val recall 0.471 | test recall 0.250
Epoch 015 | loss 0.1287 | val acc 0.934 | test acc 0.901 | val recall 0.471 | test recall 0.250
Epoch 020 | loss 0.1115 | val acc 0.929 | test acc 0.897 | val recall 0.471 | test recall 0.292
Epoch 025 | loss 0.0985 | val acc 0.929 | test acc 0.893 | val recall 0.471 | test recall 0.250
Epoch 030 | loss 0.0886 | val acc 0.924 | test acc 0.901 | val recall 0.471 | test recall 0.250
